# RAG Evaluation

So far, we built a full RAG pipeline including embedding, retrieval, and generation. Now we need to assess if the system is actually working well.

RAG systems must be evaluated across multiple components, not just final answers
* Retrieval quality: Did we retrieve the right documents?
* Context quality: Is the retrieved context relevant and sufficient?
* Generation quality: Is the generated answer actually correct?


Let's start by quickly building the RAG pipeline, and then shift the focus on evaluation

### Data Loading

The dataset contains transcripts from various meetings in a JSON format. Each meeting transcript has a unique ID. IDs are the keys of the outermost structure. 

The parent dictionary look like this:
```JSON
{
  <ID> : transcript_data
}
```

The value (`transcript_data`) for each key is again a dictionary. This `transcript_data` subdictionary contains 2 key-value pairs. 


So the dataset looks like:
```JSON
{
  <ID 1> : {
      'transcript': '',             # <whole transcript as a string>,
      'questions': [                # <list of questions related to the meeting asked afterwards as a string, along with their answers>,
                    {'question': <question_sstring>, 'answer': <answer_string>},
                    {<question2 and answer2>}, <more question dicts>... 
                  ]
      },
  <ID 2> : { <transcript and question-answers>
    ...
  }
  ..
}
```


We will create two datasets from this. One will be the document corpus containing all the transcript texts, and the other will be an evaluation set containing questions, sample answers (we will evaluate generated answers against these), and meeting IDs (we will use these to check if the correct chunk is retrieved or not).


Now, we are using a small sample of this set (`meeting_transcripts.json`), being around 10 MB. You can use the file `meeting_transcripts_full.json` (~80 MB) if you want to experiment with handling large files in a RAG pipeline. It will include optimisation of cleaning, chunking, embedding, and searching processes for such huge corpora.

In [ ]:
import json

# Load the cleaned JSON file
with open('data/meeting_transcripts.json', 'r') as file:
    data = json.load(file)

Let's understand the data

In [2]:
# number of entries
len(data)

828

We have transcripts of 828 meetings. Let's once again see how one entry looks like.

In [3]:
list(data.items())[0]

('bcb70a2e-a3a6-43fa-99e1-226689d94733',
 {'transcript': "Speaker 4: Thank you. And can we do the functions for content? Items I believe are 11, three, 14, 16 and 28, I believe.\n\nSpeaker 0: Item 11 is a communication from Council on Price recommendation to increase appropriation in the general fund group in the City Manager Department by $200 to provide a contribution to the Friends of the Long Beach Public Library. Item 12 is communication from Councilman Super Now. Recommendation to increase appropriation in the special advertising and promotion fund group and the city manager's department by $10,000 to provide support for the end of summer celebration. Item 13 is a communication from Councilman Austin. Recommendation to increase appropriation in the general fund group in the city manager department by $500 to provide a donation to the Jazz Angels . Item 14 is a communication from Councilman Austin. Recommendation to increase appropriation in the general fund group in the City Mana

Note that the data contains some missing transcripts/question entries. Let's also identify and fix them.

In [4]:
t_count, q_count = 0, 0
missing_keys = []
# Check for missing or corrupted fields
for key, entry in data.items():
    if not entry.get('transcript'):
        # print(f"Missing transcript in entry with key: {key}")
        t_count = t_count+1
        missing_keys.append(key)
    if not entry.get('questions'):
        # print(f"Missing questions in entry with key: {key}")
        q_count = q_count+1
        missing_keys.append(key)
print(f"\nMissing entries:\nTranscripts: {t_count} \nQuestions: {q_count}")


Missing entries:
Transcripts: 0 
Questions: 27


So, out of 828 meetings, 27 have missing questions, and none of them have missing transcripts

### Document Creation and Chunking

In [5]:
from langchain_core.documents import Document
from tqdm import tqdm

In [6]:
# Creating a function for paragraph-based chunking

def paragraph_based_chunking(text: str, meeting_id: str, max_chunk_length: int = 1000):
    """
    Splits text into paragraph-based chunks by max chunk size
    """
    paragraphs = text.split("\n\n")
    chunks = []

    chunk = ""
    # apart from a meeting ID to identify the source, we will need a unique ID for each chunk as well (for DB)
    chunk_id=0

    for para in paragraphs:
        if len(chunk) + len(para) <= max_chunk_length:
            chunk = f"{chunk}\n\n{para}" if chunk else para
        else:
            chunks.append(Document(page_content=chunk, metadata={"meeting_id": meeting_id, "doc_id": f"{meeting_id}_{chunk_id}"}))
            chunk = para
            chunk_id += 1

    if chunk:
        chunks.append(Document(page_content=chunk, metadata={"meeting_id": meeting_id, "doc_id": f"{meeting_id}_{chunk_id}"}))

    return chunks

In [7]:
# Process JSON file and generate chunks using the above function

def process_json_and_generate_chunks(data, **kwargs):
    """
    Processes a JSON file and generates chunks in memory.

    Returns:
        List[Document]: List of all chunked documents.
    """
    all_chunks = []

    # Process each meeting entry
    for meeting_id, meeting_data in tqdm(data.items(), desc="Chunking Meetings"):
        transcript = meeting_data.get("transcript", "")

        chunks = paragraph_based_chunking(transcript, meeting_id, **kwargs)


        # Add chunks to the list
        all_chunks.extend(chunks)

    print(f"Total chunks generated: {len(all_chunks)}")
    return all_chunks

In [8]:
# Process JSON and generate chunks
all_chunks = process_json_and_generate_chunks(data, max_chunk_length=1000)

Chunking Meetings: 100%|██████████| 828/828 [00:00<00:00, 21038.73it/s]

Total chunks generated: 9122


In [9]:
all_chunks[0]

Document(metadata={'meeting_id': 'bcb70a2e-a3a6-43fa-99e1-226689d94733', 'doc_id': 'bcb70a2e-a3a6-43fa-99e1-226689d94733_0'}, page_content='Speaker 4: Thank you. And can we do the functions for content? Items I believe are 11, three, 14, 16 and 28, I believe.')

### Embeddings and Vector DB

Let's load the chunks to a Chroma collection

In [113]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    collection_name="transcripts",
    persist_directory="./chromadb_eval",
    documents=all_chunks,
    ids=[doc.metadata['doc_id'] for doc in all_chunks],
)

In [ ]:
# # get the collection (for subsequent runs)
# from langchain_chroma import Chroma

# vector_store = Chroma(
#     collection_name="transcripts",
#     persist_directory="./chromadb_eval"
# )

In [ ]:
# Adding all documents at once will be a large task
# instead, just create the store first

#  vector_store = Chroma(
#     collection_name="transcripts",
#     persist_directory="./chromadb_eval")


# Use batching to perform chunk embedding + loading

# total_chunks_added = 0
# for i in tqdm(range(0, len(chunks), batch_size), desc="Adding Chunks to DB"):
#     batch = chunks[i:i + batch_size]
#     vector_store.add_documents(batch)
# 
#     total_chunks_added += len(batch)

# print(f"Total chunks added: {total_chunks_added}")

### RAG Chain

Let's create a retriever, augment the fetched context, and create the final generation chain

```text
Query
  ↓
Dense Retriever ─┐
                 ├─→ Fusion (RRF) → Rerank → Context
BM25 Retriever ──┘
  ↓
Prompt Template
  ↓
LLM
  ↓
Response
```

#### Hybrid Retrieval

Create dense and sparse retrievers and add a score fusion function

We are just going to use an `inputs = {query, documents}` style dict to better manage the chain process. There is no absolute need to create a LangChain runnable chain everytime, but as we will see, it makes the overall process (and evaluation) a bit easier.

In [12]:
dense_retriever = vector_store.as_retriever(search_kwargs = {'k': 15})

In [13]:
from langchain_community.retrievers import BM25Retriever
bm25_retriever = BM25Retriever.from_documents(all_chunks, k=15)

In [14]:
def reciprocal_rank_fusion(results_list, k=60):
    scores = {}

    for results in results_list:
        for rank, doc in enumerate(results):
            doc_id = doc.page_content  # or use metadata["id"] if added

            if doc_id not in scores:
                scores[doc_id] = 0

            scores[doc_id] += 1 / (k + rank + 1)

    # sort by score
    ranked_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    # map back to documents
    doc_map = {doc.page_content: doc for results in results_list for doc in results}

    return [doc_map[doc_id] for doc_id, _ in ranked_docs]

In [15]:
def hybrid_search(inputs):
    query = inputs['query']

    dense_docs = dense_retriever.invoke(query)
    bm25_docs = bm25_retriever.invoke(query)
    
    fused_docs = reciprocal_rank_fusion([dense_docs, bm25_docs])
    return {"query": query, "documents": fused_docs}

And now, let's use a `RunnableLambda` to create a runnable of this part of the chain

In [16]:
from langchain_core.runnables import RunnableLambda
hybrid_retriever = RunnableLambda(hybrid_search)

#### Re-ranking

In [17]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
def rerank(inputs, top_k=5):
    query = inputs["query"]
    documents = inputs["documents"]

    pairs = [(query, doc.page_content) for doc in documents]
    
    scores = reranker.predict(pairs)
    
    scored_docs = list(zip(documents, scores))
    
    # sort by score (descending)
    ranked = sorted(scored_docs, key=lambda x: x[1], reverse=True)
    
    return {"query": query, "documents": [doc for doc, _ in ranked[:top_k]]}
    
reranking = RunnableLambda(rerank)

#### Context Augmentation and Response Synthesis

In [19]:
def format_docs(inputs):
    docs = inputs["documents"]
    context = "\n\n".join(doc.page_content for doc in docs)

    return {
        "context": context,
        "question": inputs["query"]
    }

formatter = RunnableLambda(format_docs)

In [20]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the question using ONLY the context below.

Context: {context}

Question: {question}
""")

In [21]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4.1-nano") # use any LLM

In [22]:
from langchain_core.runnables import RunnableLambda

chain = (
    RunnableLambda(lambda x: {"query": x})  # initial mapping
    | hybrid_retriever      # 15 docs
    | reranking             # final 5
    | formatter
    | prompt
    | llm
)

In [131]:
response = chain.invoke("How is the budget hearing structured for topics like fire, police, and parks?")

In [133]:
response.content

'The budget hearing for topics like fire, police, and parks is structured to first hear all of the presentations at once, then address all the questions together, followed by public comment.'

The answer to this was: *'The budget hearing for fire, police, and parks presentations will occur simultaneously, and all questions will be addressed after the presentations are concluded.'* This matches with the generated answer.


So, we are done with the complete RAG pipeline. Let's proceed to evaluation now.

## **Evaluation**

#### Ground Truth Data

Let's load the questions and answers for evaluation

This gives us fixed queries and expected answers to use as benchmarks. This is called **ground truth**.

For this, we will extract three lists - questions, answers, and the meeting ID where these are taken from. The meeting ID will be used to check if the correct transcript was used. Note that one meeting can have multiple question-answer pairs, so meeting ID list will not be completely unique values.


In [23]:
# Let's also extract the queries and answers to lists

questions = []
answers = []
ids = []

for meeting_id, content in data.items():
    for qa in content["questions"]:
        questions.append(qa["question"])
        answers.append(qa["answer"])
        ids.append(meeting_id)

len(questions) == len(answers) == len(ids), len(questions)

(True, 3468)

### Basic Metrics for Retrieval Evaluation

#### Recall@k

Checks whether the correct document is present in top-k

We retrieve the documents first and than tally if the ground truth ID is present in the retrieved chunks.

In [24]:
def recall_at_k(retriever, queries, ground_truth_ids, k=10):
    hits = 0
    
    for query, gt_id in zip(queries, ground_truth_ids):
        output = retriever.invoke(query)
        docs = output["documents"]
        
        retrieved_ids = [doc.metadata.get("meeting_id") for doc in docs[:k]]
        
        if gt_id in retrieved_ids:
            hits += 1
            
    return hits / len(queries)

#### MRR (Mean Reciprocal Rank)

Checks how early the correct document appears in the retrieved docs

In [25]:
def mean_reciprocal_rank(retriever, queries, ground_truth_ids):
    total_score = 0
    
    for query, gt_id in zip(queries, ground_truth_ids):
        output = retriever.invoke(query)
        docs = output["documents"]
        
        for rank, doc in enumerate(docs):
            if doc.metadata.get("meeting_id") == gt_id:
                total_score += 1 / (rank + 1)
                break
                
    return total_score / len(queries)

We are only testing retrievals for now. So let's create a retriever-only chain which gives the re-ranked retrieved chunks.

In [28]:
retriever_chain = (RunnableLambda(lambda x: {"query": x}) | hybrid_retriever | reranking)
retriever_chain.invoke("How is the budget hearing structured for topics like fire, police, and parks?")

{'query': 'How is the budget hearing structured for topics like fire, police, and parks?',
 'documents': [Document(metadata={'meeting_id': 'bcb70a2e-a3a6-43fa-99e1-226689d94733', 'doc_id': 'bcb70a2e-a3a6-43fa-99e1-226689d94733_3'}, page_content="Speaker 4: Thank you. That concludes the consent. Just a couple announcements for the regular agenda. So we do have a very long and full agenda today. We have the budget hearing, which will happen first and then right after the budget hearing. We have a variety of other hearings as it relate to the our local control program and sales tax agreement. And then we have we're going to go right into some issues around and bonds around the aquarium and also the second reading of the health care worker ordinance, which we're going to try to do all of that towards the beginning of the agenda. And then we have a long agenda for the rest of of the council. So I just want to warn folks that we do have a we do have a long meeting. We're going to go right in

In [156]:
# Let's test these values for the first 10 questions
print("Recall@5:", recall_at_k(retriever_chain, questions[:10], ids[:10], k=5))
print("MRR:", mean_reciprocal_rank(retriever_chain, questions[:10], ids[:10]))

Recall@5: 0.6
MRR: 0.5333333333333334


* Recall@5 → In 60% of queries, at least one chunk from the correct meeting appears in top 5
* MRR → When relevant content is retrieved, it is often ranked relatively high

Retrieval is moderately effective and ranking is reasonably good but not optimal. Some more tweaks can probably be made in the hybrid search + re-ranking stage.


These metrics assume one correct document (transcript) and exact ID match. In practice, multiple documents may be valid and relevance is semantic. Moreover, meeting transcripts can be very generic at some places and very specific at others. Some questions may be answerable from multiple meetings or overlapping content.


So we move to LLM-based evaluation, which checks for context relevance, answer correctness, groundedness, and faithfulness.

---
### RAGAS (LLM-based Evaluation)

RAGAS (Retrieval-Augmented Generation Assessment) is an open-source framework specifically designed to evaluate and quantify the performance of Retrieval-Augmented Generation (RAG) pipelines

While building a basic RAG system is relatively simple, making it production-ready is difficult because the Retriever and the Generator can fail in different ways

[RAGAS evaluates RAG systems](https://docs.ragas.io/en/stable/references/evaluate/) using semantic and contextual metrics, not just exact matches

In [ ]:
# !pip install ragas jsonref
# !pip install datasets

##### Core Metrics in RAGAS
1. **Context Recall** <br>
    Are relevant facts present in retrieved context?

2. **Context Precision**<br>
    Is retrieved context relevant (or noisy)?

3. **Faithfulness**<br>
    Is the answer grounded in the context?

4. **Answer Relevancy**<br>
    Does the answer address the query?

##### Prepare Evaluation Data

For a quick example, let's take only 10 random queries. Running evaluation for all queries will take a large amount of time and LLM usage.

RAGAS expects a structured dataset:
```Python
{
    "user_input": str,                  # the eval query
    "response": str,                    # model-generated answer
    "retrieved_contexts": List[str],    # retrieved chunks
    "reference": str                    # reference answer
}
```

Let's build the RAGAS pipeline and populate these values. 

We will
1. sample queries
2. run retrieval + generation
3. format data
4. evaluate using RAGAS

This [RAGAS guide](https://docs.ragas.io/en/stable/howtos/applications/evaluate-and-improve-rag/) explains the process in detail

In [26]:
import random
sampler = random.sample(range(len(questions)), 10)

sample_Qs = [questions[sample_id] for sample_id in sampler]
sample_GTs = [answers[sample_id] for sample_id in sampler]

In [36]:
eval_data = []

for i, q in enumerate(sample_Qs):
    
    # 1. Retrieve (for adding 'context')
    fetched_chunks = retriever_chain.invoke(q)
    contexts = [doc.page_content for doc in fetched_chunks['documents']]
    
    # 2. Generate (for adding 'answer')
    answer = chain.invoke(q)
    
    # build eval data for each question
    eval_data.append({
        "user_input": q,
        "response": answer.content,
        "retrieved_contexts": contexts,
        "reference": sample_GTs[i]
    })

In [38]:
# Create the EvaluationDataset
from ragas import EvaluationDataset
ragas_dataset = EvaluationDataset.from_list(eval_data)

In [39]:
ragas_dataset

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response', 'reference'], len=10)

#### Metrics

The evaluation dataset is ready. Now we need metrics to measure RAG performance. RAGAS provides various metrics that can be used based on your use case. You can find more about the available ones [here](https://docs.ragas.io/en/stable/concepts/metrics/). 

Note that evaluation and metric are different terms. Evaluation is the process of measuring the performance, accuracy, and relevance of your results. A metric is a dimension over which you measure these qualities (e.g., factuality or helpfulness). 

While metrics may be commonly used for a generic set of tasks, in some cases the rubrics which we use to grade these metrics may vary. Like helpfulness might mean different things for a support chatbot and a shopping assistant.

So, we can also define our own metrics using [`DiscreteMetric`](https://docs.ragas.io/en/stable/howtos/applications/evaluate-and-improve-rag/#set-up-metrics-for-rag-evaluation). For this, we need to define the metric name, prompt (scoring guidelines for LLM), and allowed values (pass, fail, etc.)

---

##### LLMs and Embeddings

RAGAS internally uses an LLM for metrics such as:
* Faithfulness → checks if answer is grounded in context
* Answer Relevancy → checks if answer addresses the question
* (some variants of) Context Precision

These are implemented as prompted evaluation tasks, not rule-based scoring.

RAGAS tries to find an LLM client to use (OpenAI usually). However, this may not always work perfectly due to environment setup. So optionally, we can also define what LLM and embedding model to use.

In [40]:
# For example, if you want to use Google

from google import genai
from ragas.llms import llm_factory
import os

# client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY1"))                          # put API keys in .env
# ragas_llm = llm_factory("gemini-2.5-flash-lite", provider="google", client=client)        # for providers other than OpenAI, it is good to define 'provider' to avoid problems

Some metrics also need embeddings (e.g., `AnswerCorrectness`)

In [ ]:
# embeddings
# from ragas.embeddings import GoogleEmbeddings
# from ragas.embeddings.base import embedding_factory

# Option A: Using embedding factory
# client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY1"))
# embeddings = embedding_factory("google", model="gemini-embedding-001", client=client)

# Option B: Auto-import (creates client automatically)
# embeddings = GoogleEmbeddings(model="gemini-embedding-001")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [43]:
from openai import OpenAI
client = OpenAI()
llm = llm_factory("gpt-4o-mini", client=client)

---
##### The `evaluate()` Method

Let's evaluate our results on:
* `context_precision` → how relevant retrieved context is
* `context_recall` → whether important info is present
* `faithfulness` → answer grounded in context

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    Faithfulness,
    ContextPrecision,
    ContextRecall
)

In [47]:
metrics = [
    ContextPrecision(llm=llm),
    ContextRecall(llm=llm),
    Faithfulness(llm=llm)
]

# Run evaluation
results = evaluate(ragas_dataset, metrics=metrics)

results.to_pandas()

Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,reference,context_precision,context_recall,faithfulness
0,What are some additional benefits of the propo...,"[Speaker 6: Good evening, honorable mayor and ...",Some additional benefits of the proposed beach...,The beach access mat could also benefit parent...,0.700000,1.00,1.000000
1,What are the implications of relinquishing the...,"[Speaker 0: Thank you. I have two items left, ...",The implications of relinquishing the city's r...,The relinquishment could affect the city's fut...,0.583333,1.00,1.000000
2,What are the next steps to clarify the 'minist...,"[Speaker 2: Thank you, Mayor, and members of t...",The next steps to clarify the 'ministerial mis...,The council will need to confirm and ratify th...,0.950000,1.00,1.000000
3,What feedback was gathered from businesses reg...,"[Speaker 2: Thank you, Mr. Mayor. And I apprec...",The feedback gathered from businesses regardin...,Many businesses expressed that the temporary p...,1.000000,0.75,1.000000
4,What role does Vice Mayor Andrews see for his ...,[Speaker 1: Thank you. Bear with him having a ...,The provided context does not mention or indic...,Vice Mayor Andrews mentioned that he and his s...,0.000000,0.00,1.000000
5,How does the city plan to assess and update th...,"[Speaker 1: Dave Shukla, Sun File, thank you v...",The city plans to assess and update the invent...,The city's next steps include hiring a consult...,0.588889,1.00,1.000000
6,How does the current ordinance address the imp...,[Speaker 0: Thank you. Next item 27.\n\nSpeake...,The current ordinance requires proof of advers...,The transcript does not provide specific detai...,0.000000,1.00,1.000000
7,What specific challenges are small businesses ...,[Speaker 10: Honorable mayor and members of th...,The specific challenges small businesses are f...,"Many small businesses, specifically those not ...",0.583333,0.00,0.909091
8,Can we get more details on how the contributio...,"[Speaker 1: Adam, 29, is a communication from ...",The provided context does not include specific...,The transcript does not provide specific detai...,0.700000,0.00,1.000000
9,How can the council gather more feedback from ...,"[Speaker 0: Okay. Mr. Murdoch, is there a pres...",The council can gather more feedback from resi...,The council intends to conduct a robust proces...,0.887500,1.00,0.875000


So, what this shows is that the system is already quite reliable on the generation side, the answers stay grounded and avoid hallucination. 

The main limitation is in retrieval: sometimes we miss relevant context, and sometimes we retrieve extra noise. Even then, the model often compensates, which is why the answers still look reasonable. 

But for a more robust pipeline, especially at scale, the next step is to strengthen retrieval-improving recall and ranking so that the model consistently gets the right evidence to work with.

---

##### Experiment-Based Evaluation (New)

Note that the metric imports are being moved to `ragas.metrics.collections` instead of just `ragas.metrics`. This move is aimed towards more modular metrics, better control over evaluation.

The `evaluate()` function is being deprecated, which started from version 0.4.0

Though it still works, as we saw, but discouraged. It has been replaced by `@experiment()` decorator for better structured workflows. You can refer to the [migration guide here](https://docs.ragas.io/en/stable/howtos/migrations/migrate_from_v03_to_v04/?h=#evaluation-to-experiment).

For now, we used the traditional method. Import `ragas` version < 1.0 for this.

In [ ]:
# from ragas import evaluate
# from ragas.metrics import (
#     Faithfulness,
#     ContextPrecision,
#     ContextRecall
# )

This transition moves the evaluation pipeline from dataset-level to experiment-based. Instead of evaluating the dataset, the focus is now on asynchronously running the metric-centred evaluation.

So, for example, instead of creating a dataset and then proceeding to use the `evaluate()` method, we directly move to experiment evaluation

In [ ]:
# # suppose row contains questions and ground truth answers

# # Define experiment result structure
# class ExperimentResult(BaseModel):
#     faithfulness: float
#     answer_relevancy: float

# # Create experiment function

# @experiment(ExperimentResult)
# def evaluate_rag(row: Dict[str, Any], rag, llm):
#     """
#     Run RAG evaluation on a single row (synchronous example)
#     """
#     correctness_metric = 'SomeMetric'
#     question = row["question"]

#     # Query the RAG system
#     context = retriever_chain.invoke(question)

#     rag_response = chain.invoke(question)

#     model_response = rag_response.content

#     # Evaluate correctness
#     score = correctness_metric.score(
#         question=question,
#         expected_answer=row["ground_truth"],
#         response=model_response,
#         llm=llm
#     )

#     result = {
#         **row,
#         "model_response": model_response,
#         "correctness_score": score.value,
#         "correctness_reason": score.reason,
#         "retrieved_documents": context
#     }

#     return result

### Conclusion

You saw how we can use retrieval focused and generation focused metrics for evaluating RAG pipeline. Remember that for LLM-as-a-judge evaluation, you can always (and it is recommended to) devise metrics that suit your domain and objective.

To learn more about designing and using metrics, refer to [this guide](https://docs.ragas.io/en/stable/concepts/metrics/overview/)

Some of the common RAGAS metrics are:

| Metric            | If Low         | What to Fix                    |
| ----------------- | -------------- | ------------------------------ |
| Context Recall    | Missing info   | increase k / improve retrieval |
| Context Precision | Too much noise | filtering / reranking          |
| Faithfulness      | Hallucination  | prompt / reranking             |
| Answer Relevancy  | Off-topic      | retrieval + prompt             |
